# GLM lightning — explore

**Dataset:** GOES **GLM-L2-LCFA** lightning *flashes*, consolidated to **one
parquet per day** by `src/download_glm.py` (raw = ~4,320 tiny NetCDFs/day,
discarded after parsing). Lives at `/mnt/disk1/glm-data/YYYY/glm_flashes_YYYYMMDD.parquet`,
clipped to CONUS + ~5° at build time. The full 2019–2026 range is still
downloading — every cell here works with whatever days exist so far.

One row per **flash** (the clustered meteorological unit — events → groups →
flashes): `time_start`, `time_end` (UTC) | `lat`, `lon` (centroid) | `energy`
(J) | `area` (m²) | `quality_flag`.

In [ ]:
# ---------------------------------------------------------------------------
# Helpers — run this cell once.
# ---------------------------------------------------------------------------
import random
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq

GLM_DIR = Path("/mnt/disk1/glm-data")
CONUS_BBOX = (-125.0, -66.5, 24.0, 50.0)
STATES_GEOJSON = Path("/mnt/disk1/goes-data/aux/us-states.geojson")  # cached


def glm_paths(glm_dir=GLM_DIR):
    """date -> parquet path for every built day (sorted by date)."""
    return {datetime.strptime(p.stem[-8:], "%Y%m%d").date(): p
            for p in sorted(glm_dir.glob("*/glm_flashes_*.parquet"))}


def load_day(dt, glm_dir=GLM_DIR):
    """One day of flashes."""
    return pd.read_parquet(glm_dir / str(dt.year) / f"glm_flashes_{dt:%Y%m%d}.parquet")

## 1. What's built so far

In [ ]:
paths = glm_paths()
days = sorted(paths)
size_gb = sum(p.stat().st_size for p in paths.values()) / 1024 ** 3
print(f"{len(days):,} day-parquets built ({size_gb:.2f} GB on disk)")
print(f"span: {days[0]} -> {days[-1]}")
print("days per year:",
      pd.Series([d.year for d in days]).value_counts().sort_index().to_dict())

## 2. Flashes per day

Row counts come straight from parquet metadata — no data is read, so this stays
fast even with thousands of days. Expect a strong seasonal cycle (US lightning
peaks in summer).

In [ ]:
counts = pd.Series({d: pq.ParquetFile(p).metadata.num_rows
                    for d, p in paths.items()}).sort_index()
ax = counts.plot(figsize=(12, 4), lw=0.8)
ax.set_ylabel("flashes / day")
ax.set_title(f"GLM flashes per day ({len(counts):,} days built so far)")
plt.tight_layout()
print(f"total flashes: {counts.sum():,} | median/day: {counts.median():,.0f} "
      f"| max/day: {counts.max():,} on {counts.idxmax()}")

## 3. One day in detail

In [ ]:
day = counts.idxmax()       # most active built day; or random.choice(days)
fl = load_day(day)
print(f"{day}: {len(fl):,} flashes")
fl.head()

In [ ]:
# flash physics at a glance (energy in femtojoules, area in km^2)
dur_ms = (fl["time_end"] - fl["time_start"]).dt.total_seconds() * 1000
stats = pd.DataFrame({
    "duration (ms)": dur_ms.describe(),
    "energy (fJ)": (fl["energy"] * 1e15).describe(),
    "area (km^2)": (fl["area"] / 1e6).describe(),
}).T
print("quality_flag counts:", fl.quality_flag.value_counts().to_dict())
stats.round(2)

In [ ]:
# diurnal cycle — CONUS convection peaks in local afternoon (~18-24 UTC)
ax = (fl["time_start"].dt.hour.value_counts().sort_index()
      .plot.bar(figsize=(9, 3), width=0.85))
ax.set_xlabel("hour (UTC)")
ax.set_ylabel("flashes")
ax.set_title(f"diurnal cycle on {day}")
plt.tight_layout()

In [ ]:
# spatial density — where the storms were
fig, ax = plt.subplots(figsize=(11, 6))
hb = ax.hexbin(fl["lon"], fl["lat"], gridsize=120, bins="log", mincnt=1,
               cmap="inferno")
if STATES_GEOJSON.exists():
    import geopandas as gpd
    gpd.read_file(STATES_GEOJSON).boundary.plot(ax=ax, lw=0.4, color="0.6")
ax.set_xlim(CONUS_BBOX[0], CONUS_BBOX[1])
ax.set_ylim(CONUS_BBOX[2], CONUS_BBOX[3])
fig.colorbar(hb, ax=ax, label="flashes (log scale)")
ax.set_title(f"GLM flash density on {day}")
plt.tight_layout()

## Notes

- **Flashes, not events.** LCFA is a hierarchy (events → groups → flashes); the
  per-day parquets keep flashes only — the meteorological unit, ~1/22 the size.
- **Full point resolution is preserved** — no gridding yet. Aggregating flash
  counts/energy onto the 25 km CONUS grid (see `clouds_vs_floods.ipynb`) is the
  natural feature-extraction step later.
- **Deep convection link:** flash density tends to ride under the coldest IR
  cloud tops (GOES band 13) — a candidate precursor signal for flooding.
- Download more days: `uv run python src/download_glm.py build` (resumable;
  defaults to 2019-01-01 → 2026-02-28).